## Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join
import joblib
import sys
import torch
sys.path.append("../../")

from src.configs.default_configs import fn_model, fn_pred, fn_pred_perf, device
from src.configs.lung_config import data_name
from src.file_manager.filepath import FilePath
from src.models.egrue.prediction import get_eg_values, get_egRUE
from seed_file import seed
# seed = 2024

batch_size = 32
eval_batch_size = 128

tuning_seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_data_file = join(fp.get_preprocessed_folder(), "raw_data", "lungcancerdataset.csv")
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

# Load Data

In [ ]:
split_dict_scaled = joblib.load(fp_split_dict_file)
feat_cols_w_pc = ['pc1', 'pc2', 'pc3', 'age_interview', 'BMI', 'telomere length', 'Leisure screen time', 'ahei2010score', 'amed', 'dash', 'SBP', 'DBP', 'pgs000070', 'pgs000721', 'sex (0=Male, 1=Female)', 'alcohol_0_12', 'smoke_ex(1)', 'smoke_current(2)', 'prevalent diabetes']
target_col = "lung cancer"

# Get egRUE

In [ ]:
fp_decoder_predictions_file = join(fp.get_parent_folder(fn_pred), "tuning_decoder.csv")
pred_df_classifier_decoder = pd.read_csv(fp_decoder_predictions_file, index_col=0)
fp_model = join(fp.get_parent_folder(fn_model), "decoder_tuned.pt")
classifier_decoder = torch.load(fp_model, mmap=device)
eg_df = get_eg_values(classifier_decoder, split_dict_scaled, feat_cols_w_pc, seed=seed)
eg_df

In [ ]:
pred_df_egRUE = get_egRUE(pred_df_classifier_decoder, eg_df, feat_cols_w_pc)
pred_df_egRUE 

In [ ]:
fp_egRUE_predictions_file = join(fp.get_parent_folder(fn_pred), "egRUE.csv")
pred_df_egRUE.to_csv(fp_egRUE_predictions_file)